# [Step 5 - MarkdownLoader] Markdown as text: light is right

**MLCourse - Agentic AI - Module 05: Document Loaders**

> Stage in the capstone: stage 1 INGEST: the capstone reads user-supplied documents with exactly these loaders

## What you'll learn

- why markdown often needs NO dedicated loader - plain `TextLoader` wins
- that headers survive loading as ordinary content lines starting with `#`
- a real manipulation: extracting a table of contents from loaded content
- when the heavier `UnstructuredMarkdownLoader` earns its dependency cost

---

In [1]:
# =====================================================================
# CELL 1 - SHARED SETUP: imports, track discovery, download-once cache
# =====================================================================
# Standard plumbing, identical across the whole track.
# ---------------------------------------------------------------------

# --- Standard library -------------------------------------------------
import os                      # file-system odds and ends
import urllib.request          # kept for parity with sibling notebooks
from pathlib import Path       # object-oriented filesystem paths

# --- Lesson-specific imports ------------------------------------------
from langchain_community.document_loaders import TextLoader   # md-as-text route

# ---------------------------------------------------------------------
# TRACK WALKER - resolve 03_agentic_ai by walking upward from cwd.
# ---------------------------------------------------------------------
def _find_track(start: Path) -> Path:
    """Return the absolute path of the 03_agentic_ai track root."""
    for candidate in (start, *start.parents):
        hit = candidate / "03_agentic_ai"
        if hit.is_dir():
            return hit.resolve()
    raise FileNotFoundError(
        f"No directory named 03_agentic_ai found above {start} - "
        "run this notebook from somewhere inside the MLCourse repo."
    )

TRACK = _find_track(Path.cwd())     # .../MLCourse/03_agentic_ai
DATA = TRACK / "data"
DATA.mkdir(parents=True, exist_ok=True)

# ---------------------------------------------------------------------
# DOWNLOAD-ONCE HELPERS - present in every notebook for consistency,
# although THIS notebook downloads nothing: it eats course-internal food.
# ---------------------------------------------------------------------
def get_bytes(fname: str, url: str) -> bytes:
    """Return the file's bytes, downloading only on the very first call."""
    target = DATA / fname
    if target.exists() and target.stat().st_size > 0:
        payload = target.read_bytes()
        print(f"[cache] {fname}: {len(payload):,} bytes")
        return payload
    print(f"[fetch] {url}")
    request = urllib.request.Request(url, headers={"User-Agent": "MLCourse/1.0"})
    with urllib.request.urlopen(request, timeout=60) as response:
        payload = response.read()
    target.write_bytes(payload)
    print(f"[saved] {fname}: {len(payload):,} bytes")
    return payload

def get_text(fname: str, url: str, encoding: str = "utf-8-sig") -> str:
    """get_bytes + decode; drops a BOM if present."""
    return get_bytes(fname, url).decode(encoding, errors="replace")

def to_ascii(text: str) -> str:
    """Console-safe printing for arbitrary text."""
    return text.encode("ascii", errors="replace").decode("ascii")

# Matplotlib inline guard (harmless here, mandatory skeleton everywhere).
try:
    get_ipython().run_line_magic("matplotlib", "inline")
except Exception:
    pass

C:\Users\Thoyajaksha Kashyap\AppData\Local\Temp\ipykernel_74436\2817094173.py:13: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import TextLoader   # md-as-text route


## 1. Why markdown barely needs a loader

Markdown IS plain text. Its structure lives in visible characters:
`#` for H1, `##` for H2, `-` bullets, ``` fences. So a plain `TextLoader`
already preserves everything that matters - headers arrive as ordinary
lines starting with `#`, ready for any downstream code that cares.

LangChain ALSO ships `UnstructuredMarkdownLoader`, which parses markdown
into a proper document tree via the heavyweight `unstructured` package
(pip install unstructured - large native dependencies included). What do
you gain? Elements typed as Title / NarrativeText / Table / ListItem.
What does it cost? Install size, version fragility, slower loads.

**This course defaults LIGHT**: load as text now, exploit the `#` lines
with a real structure-aware splitter in module 06
(`MarkdownHeaderTextSplitter`) - zero extra dependencies, deterministic
behaviour, same downstream power for header-driven RAG.

## 2. Load THIS course's own README

No internet needed today: our specimen is the sibling module's README -
`06_chunking_strategies/README.md`. We resolve it relative to the TRACK
root discovered by the walker, so the notebook survives folder moves.

In [2]:
md_path = TRACK / "06_chunking_strategies" / "README.md"

if not md_path.exists():
    raise FileNotFoundError(
        f"Expected the chunking README next door but found nothing at:\n  {md_path}\n"
        "Keep the 06_chunking_strategies folder beside 05_document_loaders."
    )

loader = TextLoader(str(md_path), encoding="utf-8")
docs = loader.load()

print(f"Documents loaded : {len(docs)}")
print(f"metadata         : {docs[0].metadata}")
print(f"page_content     : {len(docs[0].page_content):,} chars\n")
print("--- first 350 characters ---")
print(to_ascii(docs[0].page_content[:350]))

Documents loaded : 1
metadata         : {'source': 'D:\\projects\\python\\MLCourse\\03_agentic_ai\\06_chunking_strategies\\README.md'}
page_content     : 7,451 chars

--- first 350 characters ---
# [Step 6 - Chunking Strategies] Cutting text so retrieval can win

> **MLCourse - Agentic AI - Module 06: Chunking Strategies**

> Stage in the capstone: stage 2 CHUNK: chunk quality decides retrieval accuracy in the capstone.

## What you'll learn

- why the chunk boundary - not the model - often decides RAG accuracy
- how overlap works, why it e


## 3. Headers survived - prove it

The claim was: light loading keeps markdown semantics VISIBLE inside
`page_content`. Verify by filtering lines that start with `#` - that is a
complete table of contents, extracted with one list comprehension.

In [3]:
content = docs[0].page_content
lines = content.splitlines()

header_lines = [ln for ln in lines if ln.lstrip().startswith("#")]
body_lines = [ln for ln in lines if ln and not ln.lstrip().startswith("#")]

print(f"total lines : {len(lines)}")
print(f"headers     : {len(header_lines)}")
print(f"body lines  : {len(body_lines)}")

print("\n--- extracted outline (every '#' line, verbatim) ---")
for h in header_lines:
    indent = "  " * (len(h) - len(h.lstrip("#")) - 1)   # depth from hash count
    print(f"{indent}{to_ascii(h.strip())}")

total lines : 146
headers     : 10
body lines  : 106

--- extracted outline (every '#' line, verbatim) ---
# [Step 6 - Chunking Strategies] Cutting text so retrieval can win
  ## What you'll learn
  ## 1. Why chunking decides RAG quality
  ## 2. Overlap theory
  ## 3. Size trade-offs: precision versus completeness
  ## 4. Strategy comparison matrix
  ## 5. Decision guide
  ## 6. Pitfalls
  ## 7. Contents
  ## Summary


## 4. One meaningful manipulation: build a section map

Retrieval loves addresses. From the header lines we can build a map of
section title -> position in the text, which later lets us jump straight
to (or filter for) the right part of the document.

In [4]:
section_map = {}
current_h1 = None
offset = 0
for ln in lines:
    stripped = ln.strip()
    if stripped.startswith("## ") and current_h1 is not None:
        # record first occurrence of each H2 under its parent H1
        key = f"{current_h1} > {stripped[3:]}"
        section_map.setdefault(key, offset)
    elif stripped.startswith("# ") and not stripped.startswith("##"):
        current_h1 = stripped[2:]
    offset += len(ln) + 1     # +1 restores the newline we split away

print(f"{len(section_map)} sections indexed. Sample entries:")
for key, pos in list(section_map.items())[:4]:
    print(f"  char_offset={pos:>6}  {to_ascii(key)}")

# Select fields demo: pull JUST the body text under one chosen heading.
target = next((k for k in section_map if "Pitfalls" in k), None)
if target:
    start = section_map[target]
    excerpt = content[start : start + 220]
    print(f"\n--- body preview under '{to_ascii(target)}' ---")
    print(to_ascii(excerpt))

9 sections indexed. Sample entries:
  char_offset=   229  [Step 6 - Chunking Strategies] Cutting text so retrieval can win > What you'll learn
  char_offset=   535  [Step 6 - Chunking Strategies] Cutting text so retrieval can win > 1. Why chunking decides RAG quality
  char_offset=  1390  [Step 6 - Chunking Strategies] Cutting text so retrieval can win > 2. Overlap theory
  char_offset=  2549  [Step 6 - Chunking Strategies] Cutting text so retrieval can win > 3. Size trade-offs: precision versus completeness

--- body preview under '[Step 6 - Chunking Strategies] Cutting text so retrieval can win > 6. Pitfalls' ---
## 6. Pitfalls

- **Pitfall - splitting mid-sentence in prose**: a retrieval hit that ends
  halfway through its own punchline poisons generation. Every chunking notebook
  here reports the percentage of chunks NOT endin


## 5. Light vs structured: the honest scorecard

**Light route (`TextLoader`, used here)**
- zero extra dependencies, instant loads, byte-faithful content
- headers stay as text: parseable by YOU or by module 06's splitter
- tables/lists are just characters - no typed elements

**Structured route (`UnstructuredMarkdownLoader`)**
- returns typed elements (Title, NarrativeText, Table, ListItem)
- handy when element TYPES drive your pipeline (e.g. tables only)
- costs the big `unstructured` dependency and slower, version-sensitive parsing

**Rule of thumb adopted by this track:** default light; reach for
`unstructured` only when you truly need typed non-text elements.

## Takeaway

**For markdown, `TextLoader` plus `#`-line awareness covers 90% of RAG
needs with zero new dependencies - and module 06's
`MarkdownHeaderTextSplitter` turns those surviving `#` lines into real
section-level Documents with lineage metadata.**

## Summary

- Markdown is text with visible bones; light loading keeps those bones
  intact inside `page_content`.
- One list comprehension over `startswith('#')` yields a usable TOC -
  proof that structure survived ingestion.
- A section map (title -> char offset) shows how far simple manipulation
  on loaded Documents already goes.
- `UnstructuredMarkdownLoader` exists for typed-element pipelines; adopt it
  deliberately, not by default.